In [525]:
import polars as pl
import pandas as pd 
import numpy as np
import os
import pyarrow.dataset as pads

from sklearn.metrics import mean_squared_error,root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_text
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RepeatedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

import seaborn as sns

In [ ]:
def readDataSubset(table_type, data_path = ""):
    if table_type not in ["ball-positions", "ball-events", "player-positions", 'lineups', "game-info"]:
        print("I dont think so")
        return -1
    if table_type == 'lineups' or table_type == "game_info":
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [527]:
# Read Ball Events Data
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table().to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read Lineups Data
lineups_subset = readDataSubset('lineups')
lineups = lineups_subset.to_table().to_pandas()
lineups_df = pl.from_pandas(lineups)

# Read Player Positions
player_pos_subset = readDataSubset('player-positions')
player_positions_11 = player_pos_subset.to_table(filter = ((pads.field("player_id") == 11))).to_pandas()
player_pos_11_df = pl.from_pandas(player_positions_11)

# Read Player Positions
player_positions_12 = player_pos_subset.to_table(filter = ((pads.field("player_id") == 12))).to_pandas()
player_pos_12_df = pl.from_pandas(player_positions_12)

# Read Player Positions
player_positions_2bm = player_pos_subset.to_table(filter = ((pads.field("player_id") == 4))).to_pandas()
player_pos_2bm_df = pl.from_pandas(player_positions_2bm)

# Read Player Positions
player_positions_ss = player_pos_subset.to_table(filter = ((pads.field("player_id") == 6))).to_pandas()
player_pos_ss_df = pl.from_pandas(player_positions_ss)

In [528]:
cleaned_11_df = player_pos_11_df.sort(
    ['game_string', 'play_per_game', 'timestamp']
).select(
    ['game_string', 'play_per_game', pl.col('timestamp').alias("1b_timestamp"),
    pl.col("field_x").alias("1b_field_x"), pl.col("field_y").alias("1b_field_y")]
)

cleaned_12_df = player_pos_12_df.sort(
    ['game_string', 'play_per_game', 'timestamp']
).select(
    ['game_string', 'play_per_game', pl.col('timestamp').alias("2b_timestamp"),
    pl.col("field_x").alias("2b_field_x"), pl.col("field_y").alias("2b_field_y")]
)

cleaned_10_df = player_pos_2bm_df.sort(
    ['game_string', 'play_per_game', 'timestamp']
).select(
    ['game_string', 'play_per_game', pl.col('timestamp').alias("2bm_timestamp"),
    pl.col("field_x").alias("2bm_field_x"), pl.col("field_y").alias("2bm_field_y")]
)

In [529]:
player_positions_df = pl.concat([cleaned_11_df, cleaned_12_df, cleaned_10_df], how = "diagonal").group_by(
        ["game_string", "play_per_game"]
).agg(
    pl.col("1b_field_x").mean(),
    pl.col("1b_field_y").mean(),
    pl.col("2b_field_x").mean(),
    pl.col("2b_field_y").mean(),
    pl.col("2bm_field_x").mean(),
    pl.col("2bm_field_y").mean(),
    pl.col("1b_timestamp").min(),
    pl.col("2b_timestamp").min(),
    pl.col("2bm_timestamp").min()
).with_columns(
    (pl.col("1b_field_x")).alias("on_1b"),
    (pl.col("2b_field_x")).alias("on_2b"),
    play_per_game = pl.col("play_per_game").cast(pl.Int64)
).select(
    ["game_string", "play_per_game", "2bm_field_x", "2bm_field_y", "1b_field_x", "1b_field_y", "2b_field_x", "2b_field_y", "on_1b", "on_2b", "1b_timestamp", "2b_timestamp", "2bm_timestamp"]
).sort(
    ["game_string", "play_per_game"]
)

In [530]:
player_positions_df = player_positions_df.with_columns(
        on_1b = pl.col("on_1b").cast(pl.Boolean),
        on_2b = pl.col("on_2b").cast(pl.Boolean)
)

In [531]:
player_positions_df = player_positions_df.with_columns(
    on_1b = pl.when(pl.col("on_1b").is_null()).then(pl.lit(False)).otherwise(pl.lit(True)),
    on_2b = pl.when(pl.col("on_2b").is_null()).then(pl.lit(False)).otherwise(pl.lit(True))
)

In [533]:
player_positions_df.head(10)

game_string,play_per_game,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp
str,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64
"""NA""",null,30.265505,142.224095,45.409729,79.596422,-17.915646,113.185658,true,true,null,null,null
"""y1_d061_VKA_PHD""",1,45.2952,134.77536,null,null,null,null,false,false,null,null,63614.0
"""y1_d061_VKA_PHD""",2,38.727397,139.787246,null,null,null,null,false,false,null,null,72964.0
"""y1_d061_VKA_PHD""",3,80.590907,154.868247,null,null,null,null,false,false,null,null,106614.0
"""y1_d061_VKA_PHD""",4,56.397807,142.550214,null,null,null,null,false,false,null,null,142414.0
"""y1_d061_VKA_PHD""",5,57.398725,142.006872,null,null,null,null,false,false,null,null,155864.0
"""y1_d061_VKA_PHD""",6,57.98781,142.254296,null,null,null,null,false,false,null,null,170114.0
"""y1_d061_VKA_PHD""",7,56.105501,141.718682,null,null,null,null,false,false,null,null,187114.0
"""y1_d061_VKA_PHD""",8,44.058538,142.775303,null,null,null,null,false,false,null,null,212564.0


In [534]:
player_positions_df.shape

(74043, 13)

In [535]:
relevant_game_string = player_positions_df["game_string"]
relevant_ppg = player_positions_df["play_per_game"]
# Read Lineups Data
lineups_subset = readDataSubset('lineups')
cf_position = lineups_subset.to_table(filter = ((pads.field('game_string').isin(relevant_game_string)) & 
                                                    (pads.field('play_per_game').isin(relevant_ppg)))).to_pandas()
lineups_df = pl.from_pandas(lineups)

In [536]:
lineups_df = lineups_df.select(
    ["game_string", "play_per_game", "at_bat"]
)

In [537]:
steals = player_positions_df.join(
    lineups_df, on = ["game_string", "play_per_game"], how = "left"
).sort(
    ["game_string", "play_per_game"]
).select(
    ["game_string", "play_per_game", "at_bat", "2bm_field_x", "2bm_field_y", "1b_field_x", "1b_field_y", "2b_field_x", "2b_field_y", "on_1b", "on_2b", "1b_timestamp", "2b_timestamp", "2bm_timestamp"]
)

steals.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64
"""NA""",null,null,30.265505,142.224095,45.409729,79.596422,-17.915646,113.185658,true,true,null,null,null
"""y1_d061_VKA_PHD""",1,1,45.2952,134.77536,null,null,null,null,false,false,null,null,63614.0
"""y1_d061_VKA_PHD""",2,1,38.727397,139.787246,null,null,null,null,false,false,null,null,72964.0
"""y1_d061_VKA_PHD""",3,2,80.590907,154.868247,null,null,null,null,false,false,null,null,106614.0
"""y1_d061_VKA_PHD""",4,3,56.397807,142.550214,null,null,null,null,false,false,null,null,142414.0


In [538]:
successful_steals = steals.with_columns(
    (pl.col("at_bat").shift(-1)).alias("next_at_bat"),
    (pl.col("on_1b").shift(-1)).alias("next_on_1b"),
    (pl.col("on_2b").shift(-1)).alias("next_on_2b"),
    (pl.col("1b_field_x").shift(-1)).alias("next_1b_x"),
    (pl.col("1b_field_y").shift(-1)).alias("next_1b_y"),
    (pl.col("1b_timestamp").shift(-1)).alias("next_1b_time"),
    (pl.col("2b_field_x").shift(-1)).alias("next_2b_x"),
    (pl.col("2b_field_y").shift(-1)).alias("next_2b_y"),
    (pl.col("2b_timestamp").shift(-1)).alias("next_2b_time"),
    (pl.col("2bm_field_x").shift(1)).alias("pre_2bm_field_x"),
    (pl.col("2bm_field_y").shift(1)).alias("pre_2bm_field_y"),
    (pl.col("2bm_timestamp").shift(1)).alias("pre_2bm_timestamp")
).filter(
    (pl.col("at_bat") == pl.col("next_at_bat")) &
    (pl.col("on_1b") == True) &
    (pl.col("next_on_1b") == False) &
    (pl.col("next_on_2b") == True)
).with_columns(
    (pl.lit(1).alias("steal_successful"))
)

successful_steals.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,null,null,true,false,7.063614e6,null,7.063614e6,59,false,true,null,null,null,-37.300561,91.3041,7.162614e6,-14.774333,149.264375,7.036714e6,1
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,null,null,true,false,7.028134e6,null,7.028134e6,53,false,true,null,null,null,-14.460947,120.432073,7.064384e6,61.461092,121.551838,7.006084e6,1
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,null,null,true,false,8.220315e6,null,8.220315e6,67,false,true,null,null,null,-18.06442,122.103828,8.253615e6,39.165271,147.668263,8.199265e6,1
"""y1_d064_VKA_PHD""",80,19,17.550415,138.999681,41.980089,86.782045,null,null,true,false,2.376139e6,null,2.376139e6,19,false,true,null,null,null,-15.399672,120.550503,2.414617e6,24.392722,146.35625,2.343139e6,1
"""y1_d064_VKA_PHD""",177,42,85.860747,158.754184,37.612485,90.872906,null,null,true,false,5.545297e6,null,5.545297e6,42,false,true,null,null,null,-31.334476,101.181728,5.580574e6,88.400831,156.371937,5.522923e6,1


In [539]:
print(successful_steals.shape)

(462, 27)


In [542]:
failed_steals = steals.with_columns(
    (pl.col("at_bat").shift(-1)).alias("next_at_bat"),
    (pl.col("on_1b").shift(-1)).alias("next_on_1b"),
    (pl.col("on_2b").shift(-1)).alias("next_on_2b"),
    (pl.col("1b_field_x").shift(-1)).alias("next_1b_x"),
    (pl.col("1b_field_y").shift(-1)).alias("next_1b_y"),
    (pl.col("1b_timestamp").shift(-1)).alias("next_1b_time"),
    (pl.col("2b_field_x").shift(-1)).alias("next_2b_x"),
    (pl.col("2b_field_y").shift(-1)).alias("next_2b_y"),
    (pl.col("2b_timestamp").shift(-1)).alias("next_2b_time"),
    (pl.col("2bm_field_x").shift(1)).alias("pre_2bm_field_x"),
    (pl.col("2bm_field_y").shift(1)).alias("pre_2bm_field_y"),
    (pl.col("2bm_timestamp").shift(1)).alias("pre_2bm_timestamp")
).filter(
    (pl.col("at_bat") == pl.col("next_at_bat")) &
    (pl.col("on_1b") == True) &
    (pl.col("next_on_1b") == False) &
    (pl.col("next_on_2b") == False)
).with_columns(
    (pl.lit(0)).alias("steal_successful")
)

failed_steals.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""y1_d061_VKA_PHD""",106,33,14.220055,134.185212,39.55026,88.716391,null,null,true,false,3.374014e6,null,3.374014e6,33,false,false,null,null,null,null,null,null,28.918463,134.453148,3.335664e6,0
"""y1_d063_VKA_PHD""",200,56,20.057146,145.635799,57.256458,70.502535,null,null,true,false,6.594565e6,null,6.594565e6,56,false,false,null,null,null,null,null,null,19.59941,145.59229,6.579765e6,0
"""y1_d065_VAS_PHD""",13,4,-13.429673,153.893242,57.667477,71.976826,null,null,true,false,257937.0,null,257937.0,4,false,false,null,null,null,null,null,null,51.523045,123.854906,221505.0,0
"""y1_d065_VAS_PHD""",62,16,12.614919,146.178639,55.797962,74.381445,null,null,true,false,1.992253e6,null,1.992253e6,16,false,false,null,null,null,null,null,null,10.950813,141.723785,1.972387e6,0
"""y1_d067.5_NLG_ANI""",129,32,49.171023,129.941915,55.800985,73.721163,null,null,true,false,4.210581e6,null,4.210581e6,32,false,false,null,null,null,null,null,null,36.494687,151.986389,4.174581e6,0


In [543]:
final_steal_df = pl.concat([successful_steals, failed_steals], how = "vertical").sort(["game_string", "play_per_game"])

final_steal_df.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""y1_d061_VKA_PHD""",106,33,14.220055,134.185212,39.55026,88.716391,null,null,true,false,3.374014e6,null,3.374014e6,33,false,false,null,null,null,null,null,null,28.918463,134.453148,3.335664e6,0
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,null,null,true,false,7.063614e6,null,7.063614e6,59,false,true,null,null,null,-37.300561,91.3041,7.162614e6,-14.774333,149.264375,7.036714e6,1
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,null,null,true,false,7.028134e6,null,7.028134e6,53,false,true,null,null,null,-14.460947,120.432073,7.064384e6,61.461092,121.551838,7.006084e6,1
"""y1_d063_VKA_PHD""",200,56,20.057146,145.635799,57.256458,70.502535,null,null,true,false,6.594565e6,null,6.594565e6,56,false,false,null,null,null,null,null,null,19.59941,145.59229,6.579765e6,0
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,null,null,true,false,8.220315e6,null,8.220315e6,67,false,true,null,null,null,-18.06442,122.103828,8.253615e6,39.165271,147.668263,8.199265e6,1


In [544]:
print(final_steal_df.shape)
final_steal_df.describe()

(654, 27)


statistic,game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""654""",654.0,654.0,654.0,654.0,654.0,654.0,124.0,124.0,654.0,654.0,654.0,124.0,654.0,654.0,654.0,654.0,0.0,0.0,0.0,462.0,462.0,462.0,654.0,654.0,654.0,654.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,530.0,530.0,0.0,0.0,0.0,530.0,0.0,0.0,0.0,0.0,654.0,654.0,654.0,192.0,192.0,192.0,0.0,0.0,0.0,0.0
"""mean""",null,172.46789,43.310398,27.45945,139.01304,41.38501,86.527323,-25.159411,107.761381,1.0,0.189602,6.4517e6,6.8663e6,6.4517e6,43.310398,0.0,0.706422,null,null,null,-16.293404,117.320944,6.9595e6,30.383798,141.557941,6.4223e6,0.706422
"""std""",null,97.910692,24.240689,21.215932,8.021234,11.744916,8.84901,8.854024,11.155585,null,null,7.2899e6,6.9331e6,7.2899e6,24.240689,null,null,null,null,null,6.698997,11.346148,8.3215e6,20.993214,9.229608,7.2896e6,0.455749
"""min""","""y1_d061_VKA_PHD""",2.0,1.0,-27.005342,98.333135,-8.307622,67.870861,-38.821334,75.078793,1.0,0.0,39061.0,507983.0,39061.0,1.0,0.0,0.0,null,null,null,-37.300561,57.524072,65511.0,-26.591278,105.542686,6411.0,0.0
"""25%""",null,85.0,22.0,12.280979,134.915542,36.421364,77.962281,-32.022855,99.215682,null,null,2.837047e6,3.613582e6,2.837047e6,22.0,null,null,null,null,null,-16.754145,117.886604,3.163686e6,17.331803,136.335566,2.796745e6,0.0
"""50%""",null,173.0,44.0,23.679444,139.403366,39.799079,89.003262,-29.140982,104.673675,null,null,5.788966e6,6.600119e6,5.788966e6,44.0,null,null,null,null,null,-14.507518,120.558865,6.096816e6,27.320903,142.188765,5.715412e6,1.0
"""75%""",null,249.0,63.0,43.679096,143.251537,52.939475,92.505224,-15.186775,119.344206,null,null,8.535143e6,8.624032e6,8.535143e6,63.0,null,null,null,null,null,-12.899759,122.704296,8.791293e6,45.770691,147.54412,8.511993e6,1.0
"""max""","""y1_d217_VAS_ARN""",399.0,98.0,107.366754,169.493268,61.043127,111.012331,-10.390876,129.063604,1.0,1.0,7.5139163e7,7.5139163e7,7.5139163e7,98.0,0.0,1.0,null,null,null,36.466698,129.191683,7.5163613e7,106.891308,202.253419,7.5116163e7,1.0


In [547]:
catcher_events_to_2bm = ball_events_df.sort(
    ["game_string", "play_per_game", "timestamp"]
).filter(
    (pl.col("player_id") == 1) &
    (pl.col("ball_eventcode") == 1) &
    (pl.col("player_id").shift(-1) == 2) &
    (pl.col("ball_eventcode").shift(-1) == 2) &
    (pl.col("player_id").shift(-2) == 2) &
    (pl.col("ball_eventcode").shift(-2) == 3) &
    (pl.col("player_id").shift(-3) == 4) &
    (pl.col("ball_eventcode").shift(-3) == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("throw_time"), pl.lit(1).alias("2bm")]
)

ball_events_2bm = ball_events_df.sort(
    ["game_string", "play_per_game", "timestamp"]
).filter(
    (pl.col("player_id").shift(1) == 2) &
    (pl.col("ball_eventcode").shift(1) == 3) &
    (pl.col("player_id") == 4) &
    (pl.col("ball_eventcode") == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("catch_time"), pl.lit(1).alias("2bm")]
)

catcher_events_to_ss = ball_events_df.sort(
    ["game_string", "play_per_game", "timestamp"]
).filter(
    (pl.col("player_id") == 1) &
    (pl.col("ball_eventcode") == 1) &
    (pl.col("player_id").shift(-1) == 2) &
    (pl.col("ball_eventcode").shift(-1) == 2) &
    (pl.col("player_id").shift(-2) == 2) &
    (pl.col("ball_eventcode").shift(-2) == 3) &
    (pl.col("player_id").shift(-3) == 6) &
    (pl.col("ball_eventcode").shift(-3) == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("throw_time"), pl.lit(1).alias("ss")]
)

ball_events_ss = ball_events_df.sort(
    ["game_string", "play_per_game", "timestamp"]
).filter(
    (pl.col("player_id").shift(1) == 2) &
    (pl.col("ball_eventcode").shift(1) == 3) &
    (pl.col("player_id") == 6) &
    (pl.col("ball_eventcode") == 2)
    .over(["game_string", "play_per_game"])
).select(
    ["game_string", "play_per_game", pl.col("timestamp").alias("catch_time"), pl.lit(1).alias("ss")]
)

In [548]:
catcher_steals = pl.concat([catcher_events_to_2bm, ball_events_2bm, catcher_events_to_ss, ball_events_ss], how = "diagonal").group_by(
    ["game_string", "play_per_game", "2bm", "ss"]
).agg(
    pl.col("throw_time").min(),
    pl.col("catch_time").min()
).with_columns(
    throw_time = ((pl.col("catch_time") - pl.col("throw_time")) / 1000).round(2),
    play_per_game = pl.col("play_per_game").cast(pl.Int64)
).sort(
    ["game_string", "play_per_game"]
).select(
    ["game_string", "play_per_game", "throw_time", "2bm", "ss"]
).drop_nulls("play_per_game").drop_nulls("throw_time").fill_null(0)

catcher_steals.head()

game_string,play_per_game,throw_time,2bm,ss
str,i64,f64,i32,i32
"""y1_d061_VKA_PHD""",106,2.35,1,0
"""y1_d064_VKA_PHD""",177,2.57,0,1
"""y1_d065_VAS_PHD""",80,2.41,1,0
"""y1_d070_SQF_ARN""",167,2.38,0,1
"""y1_d071_SQF_ARN""",278,2.31,0,1


In [551]:
catcher_steals.sample(n = 10)

game_string,play_per_game,throw_time,2bm,ss
str,i64,f64,i32,i32
"""y1_d201_JGO_VAS""",76,2.65,0,1
"""y1_d211_PHD_VAS""",59,2.9,0,1
"""y1_d168_MPC_PHD""",58,2.4,1,0
"""y1_d162_KZR_VAS""",59,2.4,1,0
"""y1_d166_SZJ_ANI""",183,2.45,0,1
"""y1_d124_VAS_PHD""",93,2.4,0,1
"""y1_d167_MPC_PHD""",11,2.4,0,1
"""y1_d148_VHA_PHD""",202,2.34,0,1
"""y1_d179_YXL_PHD""",195,2.6,0,1


In [458]:
player_positions_2bm

,game_string,play_per_game,timestamp,player_id,field_x,field_y,home_team,away_team,year,day
0,y1_d143_ADQ_ANI,1.0,50804.0,4,41.1609,148.5990,ANI,ADQ,y1,d143
1,y1_d143_ADQ_ANI,1.0,50854.0,4,41.1546,148.5960,ANI,ADQ,y1,d143
2,y1_d143_ADQ_ANI,1.0,50904.0,4,41.1480,148.5927,ANI,ADQ,y1,d143
3,y1_d143_ADQ_ANI,1.0,50954.0,4,41.1414,148.5897,ANI,ADQ,y1,d143
4,y1_d143_ADQ_ANI,1.0,51004.0,4,41.1345,148.5867,ANI,ADQ,y1,d143
...,...,...,...,...,...,...,...,...,...,...
7617753,y1_d103_WTH_VAS,306.0,10163985.0,4,-5.6718,148.5492,VAS,WTH,y1,d103
7617754,y1_d103_WTH_VAS,306.0,10164035.0,4,-5.6442,148.2768,VAS,WTH,y1,d103
7617755,y1_d103_WTH_VAS,306.0,10164085.0,4,-5.6163,148.0020,VAS,WTH,y1,d103
7617756,y1_d103_WTH_VAS,306.0,10164135.0,4,-5.5881,147.7260,VAS,WTH,y1,d103


In [460]:
player_pos_2bm_df.head()

game_string,play_per_game,2bm,timestamp,field_x,field_y,pre_ppg,pre_field_x,pre_field_y,pre_timestamp
str,i64,i32,f64,f64,f64,i64,f64,f64,f64
"""NA""",null,1,null,30.265505,142.224095,null,null,null,null
"""y1_d061_VKA_PHD""",1,1,63614.0,45.2952,134.77536,null,30.265505,142.224095,null
"""y1_d061_VKA_PHD""",2,1,72964.0,38.727397,139.787246,1,45.2952,134.77536,63614.0
"""y1_d061_VKA_PHD""",3,1,106614.0,80.590907,154.868247,2,38.727397,139.787246,72964.0
"""y1_d061_VKA_PHD""",4,1,142414.0,56.397807,142.550214,3,80.590907,154.868247,106614.0


In [552]:
player_pos_2bm_df = player_pos_2bm_df.with_columns(play_per_game = pl.col("play_per_game").cast(pl.Int64)).select(
    ["game_string", "play_per_game", "timestamp", "field_x", "field_y", pl.lit(1).alias("2bm")]
)
player_pos_ss_df = player_pos_ss_df.with_columns(play_per_game = pl.col("play_per_game").cast(pl.Int64)).select(
    ["game_string", "play_per_game", "timestamp", "field_x", "field_y", pl.lit(1).alias("ss")]
)

In [553]:
player_pos_2bm_df.head()

game_string,play_per_game,timestamp,field_x,field_y,2bm
str,i64,f64,f64,f64,i32
"""y1_d143_ADQ_ANI""",1,50804.0,41.1609,148.599,1
"""y1_d143_ADQ_ANI""",1,50854.0,41.1546,148.596,1
"""y1_d143_ADQ_ANI""",1,50904.0,41.148,148.5927,1
"""y1_d143_ADQ_ANI""",1,50954.0,41.1414,148.5897,1
"""y1_d143_ADQ_ANI""",1,51004.0,41.1345,148.5867,1


In [554]:
player_pos_2bm_df = player_pos_2bm_df.group_by(["game_string", "play_per_game", "2bm"]
).agg(
    pl.col("timestamp").min(),
    pl.col("field_x").mean(),
    pl.col("field_y").mean()
).sort(["game_string", "play_per_game"])

player_pos_ss_df = player_pos_ss_df.group_by(["game_string", "play_per_game", "ss"]
).agg(
    pl.col("timestamp").min(),
    pl.col("field_x").mean(),
    pl.col("field_y").mean()
).sort(["game_string", "play_per_game"])

In [555]:
player_pos_2bm_df = player_pos_2bm_df.with_columns(
    pl.col("play_per_game").shift(1).alias("pre_ppg"), 
    pl.col("field_x").shift(1).alias("pre_field_x"),
    pl.col("field_y").shift(1).alias("pre_field_y"),
    pl.col("timestamp").shift(1).alias("pre_timestamp")
)
player_pos_2bm_df

game_string,play_per_game,2bm,timestamp,field_x,field_y,pre_ppg,pre_field_x,pre_field_y,pre_timestamp
str,i64,i32,f64,f64,f64,i64,f64,f64,f64
"""NA""",null,1,null,30.265505,142.224095,null,null,null,null
"""y1_d061_VKA_PHD""",1,1,63614.0,45.2952,134.77536,null,30.265505,142.224095,null
"""y1_d061_VKA_PHD""",2,1,72964.0,38.727397,139.787246,1,45.2952,134.77536,63614.0
"""y1_d061_VKA_PHD""",3,1,106614.0,80.590907,154.868247,2,38.727397,139.787246,72964.0
"""y1_d061_VKA_PHD""",4,1,142414.0,56.397807,142.550214,3,80.590907,154.868247,106614.0
…,…,…,…,…,…,…,…,…,…
"""y1_d217_VAS_ARN""",347,1,1.1134986e7,18.670318,143.765945,346,19.085584,143.88822,1.1111436e7
"""y1_d217_VAS_ARN""",348,1,1.1176636e7,18.78218,143.409207,347,18.670318,143.765945,1.1134986e7
"""y1_d217_VAS_ARN""",349,1,1.1215536e7,-9.416407,137.521095,348,18.78218,143.409207,1.1176636e7


In [556]:
player_pos_ss_df = player_pos_ss_df.with_columns(
    pl.col("play_per_game").shift(1).alias("pre_ppg"), 
    pl.col("field_x").shift(1).alias("pre_field_x"),
    pl.col("field_y").shift(1).alias("pre_field_y"),
    pl.col("timestamp").shift(1).alias("pre_timestamp")
)

player_pos_ss_df.head()

game_string,play_per_game,ss,timestamp,field_x,field_y,pre_ppg,pre_field_x,pre_field_y,pre_timestamp
str,i64,i32,f64,f64,f64,i64,f64,f64,f64
"""NA""",null,1,null,-28.898772,140.343837,null,null,null,null
"""y1_d061_VKA_PHD""",1,1,63614.0,-26.56044,147.14934,null,-28.898772,140.343837,null
"""y1_d061_VKA_PHD""",2,1,72964.0,-18.536207,147.363898,1,-26.56044,147.14934,63614.0
"""y1_d061_VKA_PHD""",3,1,106614.0,30.283269,146.081511,2,-18.536207,147.363898,72964.0
"""y1_d061_VKA_PHD""",4,1,142414.0,-21.61167,152.021497,3,30.283269,146.081511,106614.0


In [557]:
ss_2bm_pos = pl.concat([player_pos_2bm_df, player_pos_ss_df], how = "diagonal")

In [558]:
ss_2bm_pos.head()

game_string,play_per_game,2bm,timestamp,field_x,field_y,pre_ppg,pre_field_x,pre_field_y,pre_timestamp,ss
str,i64,i32,f64,f64,f64,i64,f64,f64,f64,i32
"""NA""",null,1,null,30.265505,142.224095,null,null,null,null,null
"""y1_d061_VKA_PHD""",1,1,63614.0,45.2952,134.77536,null,30.265505,142.224095,null,null
"""y1_d061_VKA_PHD""",2,1,72964.0,38.727397,139.787246,1,45.2952,134.77536,63614.0,null
"""y1_d061_VKA_PHD""",3,1,106614.0,80.590907,154.868247,2,38.727397,139.787246,72964.0,null
"""y1_d061_VKA_PHD""",4,1,142414.0,56.397807,142.550214,3,80.590907,154.868247,106614.0,null


In [559]:
ss_2bm_pos = ss_2bm_pos.sort(["game_string", "play_per_game"]).select(
    ["game_string", "play_per_game", "timestamp", "pre_timestamp" ,"field_x", "pre_field_x" ,"field_y", "pre_field_y"]
).drop_nulls("play_per_game")

ss_2bm_pos.head()

game_string,play_per_game,timestamp,pre_timestamp,field_x,pre_field_x,field_y,pre_field_y
str,i64,f64,f64,f64,f64,f64,f64
"""y1_d061_VKA_PHD""",1,63614.0,null,45.2952,30.265505,134.77536,142.224095
"""y1_d061_VKA_PHD""",1,63614.0,null,-26.56044,-28.898772,147.14934,140.343837
"""y1_d061_VKA_PHD""",2,72964.0,63614.0,38.727397,45.2952,139.787246,134.77536
"""y1_d061_VKA_PHD""",2,72964.0,63614.0,-18.536207,-26.56044,147.363898,147.14934
"""y1_d061_VKA_PHD""",3,106614.0,72964.0,80.590907,38.727397,154.868247,139.787246


In [560]:
catcher_steals.head()

game_string,play_per_game,throw_time,2bm,ss
str,i64,f64,i32,i32
"""y1_d061_VKA_PHD""",106,2.35,1,0
"""y1_d064_VKA_PHD""",177,2.57,0,1
"""y1_d065_VAS_PHD""",80,2.41,1,0
"""y1_d070_SQF_ARN""",167,2.38,0,1
"""y1_d071_SQF_ARN""",278,2.31,0,1


In [561]:
catcher_positions = catcher_steals.join(ss_2bm_pos, on = ["game_string", "play_per_game"], how = "left").unique(["game_string", "play_per_game"]).sort(["game_string", "play_per_game"])

catcher_positions.head()

game_string,play_per_game,throw_time,2bm,ss,timestamp,pre_timestamp,field_x,pre_field_x,field_y,pre_field_y
str,i64,f64,i32,i32,f64,f64,f64,f64,f64,f64
"""y1_d061_VKA_PHD""",106,2.35,1,0,3.374014e6,3.335664e6,-27.213194,-30.798514,134.601115,122.364275
"""y1_d064_VKA_PHD""",177,2.57,0,1,5.545297e6,5.522923e6,85.860747,88.400831,158.754184,156.371937
"""y1_d065_VAS_PHD""",80,2.41,1,0,2.584175e6,2.561966e6,13.233298,14.436205,136.519215,138.41818
"""y1_d070_SQF_ARN""",167,2.38,0,1,5.355615e6,5.324067e6,36.834054,19.612112,135.085913,149.622917
"""y1_d071_SQF_ARN""",278,2.31,0,1,9.171244e6,9.14788e6,29.055503,35.106922,137.46061,135.18895


In [562]:
final_steal_df.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""y1_d061_VKA_PHD""",106,33,14.220055,134.185212,39.55026,88.716391,null,null,true,false,3.374014e6,null,3.374014e6,33,false,false,null,null,null,null,null,null,28.918463,134.453148,3.335664e6,0
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,null,null,true,false,7.063614e6,null,7.063614e6,59,false,true,null,null,null,-37.300561,91.3041,7.162614e6,-14.774333,149.264375,7.036714e6,1
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,null,null,true,false,7.028134e6,null,7.028134e6,53,false,true,null,null,null,-14.460947,120.432073,7.064384e6,61.461092,121.551838,7.006084e6,1
"""y1_d063_VKA_PHD""",200,56,20.057146,145.635799,57.256458,70.502535,null,null,true,false,6.594565e6,null,6.594565e6,56,false,false,null,null,null,null,null,null,19.59941,145.59229,6.579765e6,0
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,null,null,true,false,8.220315e6,null,8.220315e6,67,false,true,null,null,null,-18.06442,122.103828,8.253615e6,39.165271,147.668263,8.199265e6,1


In [563]:
new_df = final_steal_df.join(catcher_positions, on = ["game_string", "play_per_game"], how = "left").with_columns(
    pl.col("field_x").fill_null(pl.col("2bm_field_x")),
    pl.col("field_y").fill_null(pl.col("2bm_field_y")),
    pl.col("pre_field_x").fill_null(pl.col("pre_2bm_field_x")),
    pl.col("pre_field_y").fill_null(pl.col("pre_2bm_field_y")),
    pl.col("timestamp").fill_null(pl.col("2bm_timestamp")),
    pl.col("pre_timestamp").fill_null(pl.col("pre_2bm_timestamp")),
    pl.col("2bm").fill_null(0),
    pl.col("ss").fill_null(0)
).sort(["game_string", "play_per_game"])

print(new_df.shape)
new_df.head()

(654, 36)


game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful,throw_time,2bm,ss,timestamp,pre_timestamp,field_x,pre_field_x,field_y,pre_field_y
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,f64,f64,f64,f64,f64,f64
"""y1_d061_VKA_PHD""",106,33,14.220055,134.185212,39.55026,88.716391,null,null,true,false,3.374014e6,null,3.374014e6,33,false,false,null,null,null,null,null,null,28.918463,134.453148,3.335664e6,0,2.35,1,0,3.374014e6,3.335664e6,-27.213194,-30.798514,134.601115,122.364275
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,null,null,true,false,7.063614e6,null,7.063614e6,59,false,true,null,null,null,-37.300561,91.3041,7.162614e6,-14.774333,149.264375,7.036714e6,1,null,0,0,7.063614e6,7.036714e6,-8.948758,-14.774333,140.726194,149.264375
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,null,null,true,false,7.028134e6,null,7.028134e6,53,false,true,null,null,null,-14.460947,120.432073,7.064384e6,61.461092,121.551838,7.006084e6,1,null,0,0,7.028134e6,7.006084e6,55.694929,61.461092,120.08158,121.551838
"""y1_d063_VKA_PHD""",200,56,20.057146,145.635799,57.256458,70.502535,null,null,true,false,6.594565e6,null,6.594565e6,56,false,false,null,null,null,null,null,null,19.59941,145.59229,6.579765e6,0,null,0,0,6.594565e6,6.579765e6,20.057146,19.59941,145.635799,145.59229
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,null,null,true,false,8.220315e6,null,8.220315e6,67,false,true,null,null,null,-18.06442,122.103828,8.253615e6,39.165271,147.668263,8.199265e6,1,null,0,0,8.220315e6,8.199265e6,30.047615,39.165271,142.709072,147.668263


In [582]:
new_df_nulls = new_df.filter(~
    (pl.col("steal_successful") == 0) &
    (pl.col("throw_time").is_null())
)

print(new_df_nulls.shape)
new_df_nulls.head()

(393, 36)


game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,2b_field_x,2b_field_y,on_1b,on_2b,1b_timestamp,2b_timestamp,2bm_timestamp,next_at_bat,next_on_1b,next_on_2b,next_1b_x,next_1b_y,next_1b_time,next_2b_x,next_2b_y,next_2b_time,pre_2bm_field_x,pre_2bm_field_y,pre_2bm_timestamp,steal_successful,throw_time,2bm,ss,timestamp,pre_timestamp,field_x,pre_field_x,field_y,pre_field_y
str,i64,i64,f64,f64,f64,f64,f64,f64,bool,bool,f64,f64,f64,i64,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,i32,i32,f64,f64,f64,f64,f64,f64
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,null,null,true,false,7.063614e6,null,7.063614e6,59,false,true,null,null,null,-37.300561,91.3041,7.162614e6,-14.774333,149.264375,7.036714e6,1,null,0,0,7.063614e6,7.036714e6,-8.948758,-14.774333,140.726194,149.264375
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,null,null,true,false,7.028134e6,null,7.028134e6,53,false,true,null,null,null,-14.460947,120.432073,7.064384e6,61.461092,121.551838,7.006084e6,1,null,0,0,7.028134e6,7.006084e6,55.694929,61.461092,120.08158,121.551838
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,null,null,true,false,8.220315e6,null,8.220315e6,67,false,true,null,null,null,-18.06442,122.103828,8.253615e6,39.165271,147.668263,8.199265e6,1,null,0,0,8.220315e6,8.199265e6,30.047615,39.165271,142.709072,147.668263
"""y1_d064_VKA_PHD""",80,19,17.550415,138.999681,41.980089,86.782045,null,null,true,false,2.376139e6,null,2.376139e6,19,false,true,null,null,null,-15.399672,120.550503,2.414617e6,24.392722,146.35625,2.343139e6,1,null,0,0,2.376139e6,2.343139e6,17.550415,24.392722,138.999681,146.35625
"""y1_d066_VAS_PHD""",27,8,66.71844,122.789625,44.685934,84.961251,null,null,true,false,3.505823e6,null,3.505823e6,8,false,true,null,null,null,-13.374544,124.306309,3.539846e6,46.671559,130.902922,3.490643e6,1,null,0,0,3.505823e6,3.490643e6,66.71844,46.671559,122.789625,130.902922


In [589]:
X = new_df_nulls.drop("steal_successful")
y = new_df_nulls["steal_successful"]

print(x.shape)
print(y.shape)

X = X.to_pandas()
y = y.to_pandas()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, train_size = .8, random_state = 42, stratify = y)

(393, 35)
(393,)


field_x = catcher_positions["field_x"].head(5)
field_y = catcher_positions["field_y"].head(5)
pre_x = catcher_positions["pre_field_x"].head(5)
pre_y = catcher_positions["pre_field_y"].head(5)
home_x = 0
home_y = 0
base_1x = 45 * np.sqrt(2)
base_1y = 45 * np.sqrt(2)
base_2x = 0
base_2y = 127.279221
base_3x = -45 * np.sqrt(2)
base_3y = 45 * np.sqrt(2)

plt.scatter(home_x, home_y, color = "white", edgecolor = "black")
plt.scatter(base_1x, base_1y, color = "white", edgecolor = "black")
plt.scatter(base_2x, base_2y, color = "white", edgecolor = "black")
plt.scatter(base_3x, base_3y, color = "white", edgecolor = "black")
plt.scatter(field_x, field_y, color = "blue")
plt.scatter(pre_x, pre_y, color = "green")
plt.xlim([-80, 100])
plt.ylim([-80, 150])

In [ ]:
dcov_df = final_steal_df.with_columns(
    # FInd the change in x, y, and time
    change_x_1b = (pl.col("next_2b_x") - pl.col("1b_field_x")).over(["game_string", "play_per_game"]),
    change_y_1b = (pl.col("next_2b_y") - pl.col("1b_field_y")).over(["game_string", "play_per_game"]),
    change_time = (pl.col("next_2b_time") - pl.col("1b_timestamp")).over(["game_string", "play_per_game"])
).with_columns(
    # convert change_time to seconds and calculate distance of step
    change_time_sec = pl.col("change_time") / 1000,
    dcov_step = (pl.col("change_x_1b") ** 2 + pl.col("change_y_1b") ** 2).sqrt().over(["game_string", "play_per_game"])
).with_columns(
    dcov_total = pl.col("dcov_step").sum().over(["game_string", "play_per_game"]),
    dcov_running = pl.col("dcov_step").cum_sum().over(["game_string", "play_per_game"])
).with_columns(
    # calculate instantaneous velo, accel, and jerk
    i_vel = pl.col("dcov_step") / pl.col("change_time_sec"),
    i_accel = (pl.col("dcov_step") / (pl.col("change_time_sec")) ** 2),
    i_jerk = (pl.col("dcov_step") / (pl.col("change_time_sec")) ** 3)
).select(
        ["game_string", "play_per_game", "at_bat", "2bm_field_x", "2bm_field_y", "1b_field_x", "1b_field_y", "next_2b_x", "next_2b_y", "change_time_sec",
         "dcov_step", "dcov_total", "dcov_running", "i_vel", "i_accel", "i_jerk", "steal_successful"]
)

In [266]:
dcov_df.head()

game_string,play_per_game,at_bat,2bm_field_x,2bm_field_y,1b_field_x,1b_field_y,next_2b_x,next_2b_y,change_time_sec,dcov_step,dcov_total,dcov_running,i_vel,i_accel,i_jerk
str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""y1_d061_VKA_PHD""",106,33,14.220055,134.185212,39.55026,88.716391,null,null,null,null,0.0,null,null,null,null
"""y1_d061_VKA_PHD""",213,59,-8.948758,140.726194,37.436083,91.688601,-37.300561,91.3041,99.0,74.737633,74.737633,74.737633,0.754926,0.007626,0.000077
"""y1_d062_VKA_PHD""",230,53,55.694929,120.08158,39.084516,89.693051,-14.460947,120.432073,36.25,61.74143,61.74143,61.74143,1.703212,0.046985,0.001296
"""y1_d063_VKA_PHD""",200,56,20.057146,145.635799,57.256458,70.502535,null,null,null,null,0.0,null,null,null,null
"""y1_d063_VKA_PHD""",253,67,30.047615,142.709072,37.423019,92.715501,-18.06442,122.103828,33.3,62.789566,62.789566,62.789566,1.885573,0.056624,0.0017


game_filter = player_positions_df.filter(
    (pl.col("game_string") == "y1_d061_VKA_PHD"),
    (pl.col("play_per_game").is_in([213, 214]))
    )

game_filter